# Multi-Voice Activity Detection — Single File Analysis

This notebook runs both the **Traditional (signal-processing)** and **DNN** MVAD models on a single WAV file and displays prediction waveforms for comparison.

In [ ]:
import sys
import numpy as np
import soundfile as sf
import scipy.io

# Import everything from mvad_test.py
# (mvad_test.py forces matplotlib backend to 'Agg' at import time,
#  so we must re-set it to 'inline' AFTER the import)
from mvad_test import (
    MultivoiceVAD,
    load_dnn_model,
    dnn_predict_file,
)

import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
%matplotlib inline
import matplotlib.pyplot as plt

print('Imports OK, matplotlib backend:', matplotlib.get_backend())

## 1. Configuration

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
INPUT_WAV   = 'inputs/example_12.wav'
# INPUT_WAV   = 'dumps/example_3_SHU1300__TMR_cfg_single_talker_mobile_with_bbldagc_48kHz_txfeOut.wav'
DNN_MODEL   = 'mvad_dnn_model_ep47.pt'
GT_MAT      = 'inputs/example_12_manualVAD.mat'   # ground truth (sample-level)

# ── Traditional VAD parameters (defaults from mvad_test.py) ───────────────
FRAME_MS              = 30
HOP_MS                = 10
ENERGY_THRESHOLD_DB   = -40
PITCH_CONF_THRESH     = 0.25
YIN_THRESHOLD         = 0.15
SECONDARY_PITCH_CONF  = 0.20
SF_THRESHOLD          = 0.30
OVERLAP_THRESHOLD     = 0.38
CONTEXT_FRAMES        = 3
MEDIAN_FILTER         = 7
F0_MIN                = 80
F0_MAX                = 400

## 2. Load audio

In [ ]:
signal, sr = sf.read(INPUT_WAV, dtype='float64')
if signal.ndim > 1:
    signal = np.mean(signal, axis=1)

duration = len(signal) / sr
print(f'File     : {INPUT_WAV}')
print(f'SR       : {sr} Hz')
print(f'Duration : {duration:.2f} s')
print(f'Samples  : {len(signal):,}')

# ── Load ground truth ────────────────────────────────────────────────────
gt_data = scipy.io.loadmat(GT_MAT)
gt_vad  = gt_data['vad'].flatten()   # sample-level: 0=silence, 1=single, 2=overlap
print(f'GT MAT   : {GT_MAT}  ({len(gt_vad):,} samples)')
for lab, name in [(0, 'Silence'), (1, 'Single'), (2, 'Overlap')]:
    cnt = np.sum(gt_vad == lab)
    print(f'  {name:20s}: {cnt/sr:.2f} s  ({cnt/len(gt_vad)*100:.1f}%)')

## 3. Run Traditional VAD

In [ ]:
vad = MultivoiceVAD(
    sr=sr,
    frame_ms=FRAME_MS,
    hop_ms=HOP_MS,
    energy_threshold_db=ENERGY_THRESHOLD_DB,
    pitch_conf_thresh=PITCH_CONF_THRESH,
    yin_threshold=YIN_THRESHOLD,
    secondary_pitch_conf=SECONDARY_PITCH_CONF,
    spectral_flatness_overlap=SF_THRESHOLD,
    overlap_threshold=OVERLAP_THRESHOLD,
    context_frames=CONTEXT_FRAMES,
    median_filter_size=MEDIAN_FILTER,
    f0_min=F0_MIN,
    f0_max=F0_MAX,
)

trad_labels, trad_features = vad.process(signal)

n_trad = len(trad_labels)
hop_s = vad.hop_len / sr
trad_times = np.arange(n_trad) * hop_s

print(f'Traditional VAD: {n_trad} frames')
for lab, name in [(0, 'Silence'), (1, 'Single'), (2, 'Overlap')]:
    cnt = np.sum(trad_labels == lab)
    print(f'  {name:20s}: {cnt:6d} frames  ({cnt/n_trad*100:5.1f}%)  {cnt*hop_s:.2f}s')

## 4. Run DNN Model

In [ ]:
model, cfg, feat_mean, feat_std, device = load_dnn_model(DNN_MODEL)

dnn_labels = dnn_predict_file(
    signal, sr, model, cfg, feat_mean, feat_std, device
)

n_dnn = len(dnn_labels)
dnn_hop_samples = cfg.get('hop_samples', int(sr * 0.01))
dnn_hop_s = dnn_hop_samples / sr
dnn_times = np.arange(n_dnn) * dnn_hop_s

print(f'DNN model: {n_dnn} frames (arch={cfg["arch"]})')
for lab, name in [(0, 'Silence'), (1, 'Single'), (2, 'Overlap')]:
    cnt = np.sum(dnn_labels == lab)
    print(f'  {name:20s}: {cnt:6d} frames  ({cnt/n_dnn*100:5.1f}%)  {cnt*dnn_hop_s:.2f}s')

## 5. Prediction Plots — Waveform + Coloured Timeline

Four panels:
1. **Audio waveform**
2. **Traditional VAD** — coloured timeline (grey=silence, green=single, red=overlap)
3. **DNN model** — coloured timeline
4. **Agreement** between both models (green=agree, red=disagree)

In [ ]:
from matplotlib.patches import Patch
from matplotlib.collections import BrokenBarHCollection

CLASS_COLORS = {0: None, 1: 'cyan', 2: 'orange'}
CLASS_ALPHA  = {0: 0.0, 1: 1.0, 2: 1.0}
CLASS_NAMES  = {0: 'Silence', 1: 'Single speaker', 2: 'Overlap'}


def _label_segments(labels, hop_sec):
    """Convert frame labels to list of (start_sec, duration_sec, label)."""
    segments = []
    if len(labels) == 0:
        return segments
    cur_label = labels[0]
    seg_start = 0.0
    for i in range(1, len(labels)):
        if labels[i] != cur_label:
            segments.append((seg_start, i * hop_sec - seg_start, cur_label))
            cur_label = labels[i]
            seg_start = i * hop_sec
    segments.append((seg_start, len(labels) * hop_sec - seg_start, cur_label))
    return segments


def _draw_timeline(ax, segments, y_bottom=0, height=1):
    """Draw coloured horizontal bars for each segment on *ax*."""
    for start, dur, lab in segments:
        if CLASS_COLORS[lab] is not None:
            ax.barh(y_bottom + height / 2, dur, height=height, left=start,
                    color=CLASS_COLORS[lab], alpha=CLASS_ALPHA[lab],
                    edgecolor='none', linewidth=0)


# ── Build segments ────────────────────────────────────────────────────────
trad_segs = _label_segments(trad_labels, hop_s)
dnn_segs  = _label_segments(dnn_labels,  dnn_hop_s)

# ── Figure ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(22, 13), sharex=True,
                         gridspec_kw={'height_ratios': [1, 1, 1],
                                      'hspace': 0.15})
fig.suptitle(f'Multivoice VAD Predictions — {INPUT_WAV}', fontsize=28, fontweight='bold', y=0.985)

# ── Panel 1: Waveform with GT fill areas ─────────────────────────────────
t_sig = np.arange(len(signal)) / sr

# Ground truth fill colours (very light, behind the waveform)
GT_FILL = {0: (None,    0.0),      # silence  — no fill
           1: ('cyan',  1.0),     # single   — cyan
           2: ('orange', 1.0)}    # overlap  — orange

# Build GT segments from sample-level labels (run-length encoding)
def _gt_sample_segments(gt, sample_rate):
    """Convert sample-level GT to (start_sec, end_sec, label) segments."""
    segs = []
    if len(gt) == 0:
        return segs
    cur = gt[0]; s0 = 0
    for i in range(1, len(gt)):
        if gt[i] != cur:
            segs.append((s0 / sample_rate, i / sample_rate, int(cur)))
            cur = gt[i]; s0 = i
    segs.append((s0 / sample_rate, len(gt) / sample_rate, int(cur)))
    return segs

gt_segs = _gt_sample_segments(gt_vad, sr)
ymin = -1.05 * np.max(np.abs(signal))
ymax =  1.05 * np.max(np.abs(signal))
for t0, t1, lab in gt_segs:
    fc, fa = GT_FILL[lab]
    if fc is not None:
        axes[0].axvspan(t0, t1, color=fc, alpha=fa, zorder=0)

axes[0].plot(t_sig, signal, lw=0.4, color='k', alpha=0.9, zorder=2)
axes[0].set_ylim(ymin, ymax)
axes[0].set_ylabel('Amplitude', fontsize=20)
axes[0].set_title('Audio Waveform (background: manual Ground Truth)', fontsize=16, loc='left')
axes[0].tick_params(axis='both', labelsize=19)
axes[0].margins(x=0)

# ── Panel 2: Traditional VAD ─────────────────────────────────────────────
_draw_timeline(axes[1], trad_segs)
axes[1].set_ylim(0, 1)
axes[1].set_yticks([])
axes[1].set_ylabel('Traditional\nVAD', fontsize=20, fontweight='bold',
                   rotation=0, labelpad=75, va='center')
axes[1].tick_params(axis='x', labelsize=19)
axes[1].margins(x=0)

# ── Panel 3: DNN ─────────────────────────────────────────────────────────
_draw_timeline(axes[2], dnn_segs)
axes[2].set_ylim(0, 1)
axes[2].set_yticks([])
axes[2].set_ylabel('DNN\nModel', fontsize=20, fontweight='bold',
                   rotation=0, labelpad=75, va='center')
axes[2].set_xlabel('Time (s)', fontsize=20)
axes[2].tick_params(axis='x', labelsize=19)
axes[2].margins(x=0)

# ── Shared legend ─────────────────────────────────────────────────────────
legend_patches = [Patch(fc='cyan', ec='none', alpha=1.0, label='Single speaker'),
                  Patch(fc='orange', ec='none', alpha=1.0, label='Overlap')]
fig.legend(handles=legend_patches, loc='upper right', ncol=2,
           framealpha=1.0, bbox_to_anchor=(0.98, 0.96),
           prop={'weight': 'bold', 'size': 20})

fig.subplots_adjust(left=0.08, right=0.98, top=0.93, bottom=0.05)
plt.show()